# 02 Symbol Optimize - Research Cockpit

This notebook optimizes parameters for **one Combo symbol** using a controlled research workflow. The goal is not to pick the single highest-Sharpe grid row, but to find a parameter region with enough sample size, reasonable drawdown, limited noise sensitivity, and full-backtest confirmation.

Main workflow:

1. Define `RUN_CONFIG`: symbol, data window, capital, costs, filter thresholds, and validation mode.
2. Audit broker specs and search space before running, so optimization does not use invalid assumptions.
3. Run fast grid search to scan a broad parameter region.
4. Filter candidates by trade count, Profit Factor, Sharpe, drawdown, and plateau stability.
5. Full-backtest the top N candidates, then select `best_params` from validation, not directly from the fast grid.
6. Read the final dashboard: KPI, monthly PnL, equity curve, trade distribution, trade explorer, and price chart.
7. For deeper robustness checks, enable walk-forward, Monte Carlo, and research export.

The final output is `best_params` plus the validation evidence. The next step is to move `best_params` into the symbol backtest or portfolio notebook to evaluate correlation, allocation, and portfolio risk.


## Notebook Reading Map

This file **does not change Combo trading rules**. It only orchestrates research to answer: for a specific symbol, which parameter region is most credible for full backtest, walk-forward, and portfolio testing?

Read in this order:

1. **Config**: define symbol, time window, capital, costs, and pass/fail criteria. Prefer changing `RUN_CONFIG` when adjusting the research setup.
2. **Spec Audit**: check broker specs, spread/swap/lot assumptions, and search space. Warnings do not always stop the notebook, but they are red flags before live use.
3. **Fast Grid**: quickly scan many parameter sets to find promising regions. Use this table for shortlisting, not final decisions.
4. **Filter + Rank**: remove weak candidates, compute robust score, and mark plateau status. Strong candidates should have decent metrics and avoid isolated spikes.
5. **Full Validation**: run full backtests for the top N. This is the decision table because it uses the detailed engine and full trade log.
6. **Final Report**: inspect KPI, equity, monthly PnL, distribution, and individual trades to understand where the edge comes from.
7. **Robustness**: walk-forward checks out-of-sample stability over time; Monte Carlo checks sequencing and tail risk.
8. **Export**: save the grid, validated candidates, and selected params for audit or portfolio-layer use.

Quick rule: if `grid` looks good but full validation is weak, discard it. If full validation looks good but has too few trades, concentrated drawdown, or weak plateau support, treat it only as a forward-test hypothesis.


In [ ]:
# Cell 1 - Bootstrap import path

import sys
from pathlib import Path


def _find_root(start: Path, marker: str = 'pyproject.toml') -> Path:
    for p in [start, *start.parents]:
        if (p / marker).exists() and (p / 'core_python' / 'shared').exists():
            return p
    raise RuntimeError(f'Could not find repo root containing {marker!r} and core_python/shared')


ROOT = _find_root(Path.cwd())
CORE = ROOT / 'core_python'
for p in (str(ROOT), str(CORE)):
    if p not in sys.path:
        sys.path.insert(0, p)

print('ROOT =', ROOT)
print('CORE =', CORE)


In [ ]:
# Cell 2 - Imports
#
#

import json
from datetime import datetime

import pandas as pd
from IPython.display import display

from core_python.shared.monte_carlo import plot_monte_carlo, run_monte_carlo
from core_python.strategies.combo.params import (
    SYMBOLS,
    get_indicator_params,
    get_symbol_params,
    get_symbol_search_space,
    summary as strategy_summary,
    validate_config,
)
from core_python.strategies.combo.model import add_combo_indicators
from core_python.strategies.combo.research_utils import (
    annotate_optimizer_plateau,
    configure_notebook,
    export_result_bundle,
    filter_optimizer_candidates,
    plot_equity_dashboard,
    plot_optimization_dashboard,
    plot_price_with_trades,
    plot_trade_distribution,
    plot_walkforward_dashboard,
    rank_optimizer_candidates,
    show_candidate_validation_report,
    show_kpi_dashboard,
    show_monthly_pnl,
    show_note,
    show_optimizer_filter_report,
    show_run_config,
    show_selected_params,
    show_trade_explorer,
    summarize_candidate_validation,
    summarize_walkforward,
    validate_symbol_candidates,
)
from core_python.strategies.combo.symbol.backtest import load_backtest_full, run_symbol_backtest
from core_python.strategies.combo.symbol.optimize import run_symbol_grid_search
from core_python.strategies.combo.symbol.walkforward import walk_forward_backtest

configure_notebook()
print(strategy_summary())
print('Symbols:', ', '.join(SYMBOLS.keys()))


## Concepts To Know Before Running

- **Fast grid**: a simplified backtest for quick parameter scanning. It helps remove bad regions, but can differ from the full engine.
- **Candidate**: a scored parameter set containing `ktp`, `x`, `ma_period`, and `trailing_activation`.
- **Filter pass**: a candidate clears minimum thresholds for sample size, PF, Sharpe, and max drawdown.
- **Robust score**: composite score that rewards return/PF/Sharpe and penalizes high drawdown, small sample size, and parameter spikes.
- **Plateau**: evidence that nearby parameter sets also work. A broad plateau is usually more credible than an isolated optimum.
- **Full validation**: detailed backtest of top candidates. `best_params` must come from this step.
- **Portfolio step**: optimizing one symbol does not automatically justify higher allocation. Allocation must consider correlation, exposure overlap, and simultaneous drawdown across symbols.


## 1. Config - Define The Optimization Problem

The next cell is the main place to edit when changing the research setup. Key parameter groups:

- `symbol`, `account_mode`, `broker_profile`: define the market and execution profile.
- `date_from`, `date_to`, `max_bars`: define the data sample. `date_to=None` means use data through the latest available bar.
- `initial_balance`, `costs`: directly affect PnL, DD, and lot sizing.
- `min_trades`, `min_profit_factor`, `max_drawdown_pct`, `min_sharpe`: minimum gates before a candidate is worth reviewing.
- `validate_top_n`: number of candidates sent to full validation. Increase it when the grid is broad or top results are close.
- `selection_mode='robust'`: select by robustness instead of a single headline metric.
- `enable_walkforward`, `enable_monte_carlo`, `export_report`: enable these for deeper research or evidence export.

Avoid optimizing too many dimensions at once when the sample is small. Each added dimension increases overfitting risk.


In [ ]:
# Cell 3 - Run config
#
# DOC_CONFIG_FIELDS
#

RUN_CONFIG = {
    'symbol': 'US30',
    'account_mode': 'standard',
    'broker_profile': None,
    'date_from': '2022-01-01',
    'date_to': None,
    'initial_balance': 100_000.0,
    'max_bars': 50_000,
    'top_n': 15,
    'validate_top_n': 10,
    'score_column': 'sharpe',
    'selection_mode': 'robust',
    'min_trades': 40,
    'min_profit_factor': 1.15,
    'max_drawdown_pct': 20.0,
    'min_sharpe': 0.5,
    'search_space_overrides': {},
    'indicator_overrides': {},
    'strategy_overrides': {},
    'costs': {},
    'enable_walkforward': False,
    'walkforward_is_bars': 5_000,
    'walkforward_oos_bars': 1_250,
    'walkforward_step_bars': 1_250,
    'enable_monte_carlo': False,
    'monte_carlo_iter': 1_000,
    'monte_carlo_dd_threshold': 0.20,
    'export_report': False,
}

if RUN_CONFIG['symbol'] not in SYMBOLS:
    raise KeyError(f"Unknown symbol {RUN_CONFIG['symbol']!r}. Available: {list(SYMBOLS)}")

show_run_config('Symbol optimize config', RUN_CONFIG)


## 2. Spec Audit - Check The Foundation Before Optimizing

Optimization is only meaningful when broker specs and search space are valid. This cell checks the active config and prints warnings for unverified assumptions.

Pay special attention to:

- `spec_verified=False`, where spread/swap/commission/contract size may not be broker-verified.
- Search space that is missing keys, too wide, or too narrow.
- Symbols missing from `SYMBOLS` or incorrect data mappings.

Errors stop the notebook. Warnings still allow research, but results should not be treated as production-ready.


In [ ]:
# Cell 4 - Broker/spec audit and search space
#
# DOC_AUDIT_PURPOSE
#

audit = validate_config(broker_profile=RUN_CONFIG.get('broker_profile'))
show_note(
    'Broker/Spec Audit',
    f"ok={audit['ok']} | warnings={len(audit['warnings'])} | errors={len(audit['errors'])}. "
    'Warnings do not block optimization, but live-oriented conclusions require verified broker specs.'
)
display(pd.DataFrame({'warnings': audit['warnings'] or ['-']}))
if audit['errors']:
    display(pd.DataFrame({'errors': audit['errors']}))
    raise RuntimeError('Config audit has errors. Fix broker/spec config before optimize.')

search_space = get_symbol_search_space(
    RUN_CONFIG['symbol'],
    broker_profile=RUN_CONFIG.get('broker_profile'),
)
search_space.update(RUN_CONFIG['search_space_overrides'])
required_keys = {'ktp', 'x', 'min_rr'}
missing = sorted(required_keys - set(search_space))
empty = sorted(k for k in required_keys if not search_space.get(k))
if missing or empty:
    raise ValueError(f'Invalid search_space. missing={missing}, empty={empty}')

show_note('Search Space', 'Grid search will test every combination below. Large grids can take a long time.')
display(pd.DataFrame([search_space]).T.rename(columns={0: 'values'}))

## 3. Fast Grid Search - Scan Broadly For Edge Regions

This cell runs `run_symbol_grid_search()` over the declared search space. The `grid` table contains one row per parameter set and its metrics.

How to read it:

- `trades`: trade count. Low counts make metrics noisy.
- `profit_factor`: gross profit / gross loss. Below-threshold values usually lack edge after costs and slippage.
- `max_drawdown_pct`: maximum capital decline. Compare it with your real risk tolerance.
- `sharpe`, `return_pct`: performance metrics that should not be used alone.

This cell guards against empty grids or zero-trade grids. If that happens, check the date window, data, session filter, spread, or overly strict entry rules.


In [ ]:
# Cell 4b - Optional: X Fill Rate Analysis
#
#
#

if RUN_CONFIG.get('analyze_x_fill_rate'):
    from core_python.strategies.combo.symbol.selection import analyze_x_fill_rate, recommend_x_range
    from core_python.strategies.combo.model import detect_combo_signals, session_mask

    sym_cfg_x = get_symbol_params(RUN_CONFIG['symbol'], broker_profile=RUN_CONFIG.get('broker_profile'))
    raw_x = load_backtest_full(sym_cfg_x['symbol_id'], max_bars=RUN_CONFIG['max_bars'])
    df_x = add_combo_indicators(raw_x.copy(), get_indicator_params())
    mask_x = session_mask(df_x, sym_cfg_x.get('session_hours_utc', []))
    df_x = detect_combo_signals(df_x, mask_x, sym_key=RUN_CONFIG['symbol'], params=get_indicator_params())

    fill_df = analyze_x_fill_rate(RUN_CONFIG['symbol'], df_x)
    x_recommended = recommend_x_range(RUN_CONFIG['symbol'], df_x)

    show_note(
        'X Fill Rate Analysis',
        f"Symbol: {RUN_CONFIG['symbol']} | Signal bars: {int((df_x.get('signal', 0) != 0).sum())} | "
        f"Recommended x: {x_recommended}"
    )
    display(fill_df)
    print(f"\nRecommended x range: {x_recommended}")
    print("To use it, add this to RUN_CONFIG:")
    print(f"  'search_space_overrides': {{'x': {x_recommended}}}")
else:
    print("X fill rate analysis is disabled. Set RUN_CONFIG['analyze_x_fill_rate'] = True to run it.")


In [ ]:
# Cell 5 - Fast grid search
#
# DOC_FAST_GRID_LIMITATION
#

grid = run_symbol_grid_search(
    RUN_CONFIG['symbol'],
    date_from=RUN_CONFIG['date_from'],
    date_to=RUN_CONFIG['date_to'],
    init_eq=RUN_CONFIG['initial_balance'],
    account_mode=RUN_CONFIG['account_mode'],
    max_bars=RUN_CONFIG['max_bars'],
    indicator_overrides=RUN_CONFIG.get('indicator_overrides') or None,
    strategy_overrides=RUN_CONFIG.get('strategy_overrides') or None,
    costs=RUN_CONFIG.get('costs') or None,
    search_space=search_space,
    broker_profile=RUN_CONFIG.get('broker_profile'),
)

if grid.empty:
    raise RuntimeError('Grid search returned no candidates.')
if 'trades' in grid.columns and int(grid['trades'].sum()) == 0:
    raise RuntimeError('Grid search produced zero trades for all candidates. Check date window, data, session, or search space.')

print('Candidates =', len(grid))
display(grid.head(RUN_CONFIG['top_n']))


## 4. Candidate Filter + Robust Ranking - Build A Disciplined Shortlist

This step turns `grid` into a validation shortlist:

1. Add plateau/stability data to see whether a candidate sits in a stable region or a spike.
2. Filter by the minimum rules in `RUN_CONFIG`.
3. Rank by `robust_score` when `selection_mode='robust'`.
4. If no candidate passes, the notebook falls back to ranking the full grid so you can inspect why. Do not treat fallback rows as passing candidates.

A candidate worth further research should have enough trades, PF above threshold, tolerable drawdown, clearly positive Sharpe, and nearby parameter support.


In [ ]:
# Cell 6 - Filter, plateau and robust ranking
#
# DOC_FILTER_RANK_DECISION
#

rules = {
    'min_trades': RUN_CONFIG['min_trades'],
    'min_profit_factor': RUN_CONFIG['min_profit_factor'],
    'max_drawdown_pct': RUN_CONFIG['max_drawdown_pct'],
    'min_sharpe': RUN_CONFIG['min_sharpe'],
}

grid_with_plateau = annotate_optimizer_plateau(
    grid,
    score_col=RUN_CONFIG['score_column'],
)
filtered_all = filter_optimizer_candidates(grid_with_plateau, rules, return_all=True)
filtered_ranked = rank_optimizer_candidates(
    filtered_all[filtered_all['filter_pass']],
    mode=RUN_CONFIG['selection_mode'],
)
if filtered_ranked.empty:
    show_note('Fallback', 'No candidates passed the filter. The notebook falls back to robust ranking across the full grid so you can inspect the cause.')
    filtered_ranked = rank_optimizer_candidates(filtered_all, mode=RUN_CONFIG['selection_mode'])

show_optimizer_filter_report(filtered_all, title='Candidate filter report')
plot_optimization_dashboard(
    filtered_ranked,
    score_col='robust_score' if 'robust_score' in filtered_ranked.columns else RUN_CONFIG['score_column'],
    top_n=RUN_CONFIG['top_n'],
)
display(filtered_ranked.head(RUN_CONFIG['top_n']))


## 5. Full Validation Top N - Decide With The Full Backtest

Fast grid is only a screening step. This cell takes the top `validate_top_n` candidates and runs the detailed full engine for each one.

The `validation_compare` table compares fast vs full results:

- If fast looks good but full is weak, the fast approximation may be too optimistic or the candidate may be execution-sensitive.
- If trade count diverges sharply, check date range, data, pending-order logic, trailing, or reversal behavior.
- If PF/Sharpe/return drops sharply, inspect costs, slippage, swap, and trade distribution.

`best_params` is selected after this table. It is the main parameter output of the notebook.


In [ ]:
# Cell 7 - Full backtest validation for top candidates
#
# DOC_VALIDATE_TOP_DECISION
#

top_candidates = filtered_ranked.head(RUN_CONFIG['validate_top_n']).copy()
validation_table, validation_results = validate_symbol_candidates(
    RUN_CONFIG['symbol'],
    top_candidates,
    RUN_CONFIG,
)
validation_compare = summarize_candidate_validation(grid_with_plateau, validation_table)
validation_compare = rank_optimizer_candidates(
    validation_compare,
    mode=RUN_CONFIG['selection_mode'],
)
show_candidate_validation_report(validation_compare)

best = validation_compare.iloc[0]
best_min_rr = best.get('min_rr')
best_params = {
    'ktp':    float(best['ktp']),
    'x':      float(best['x']),
    'min_rr': None if pd.isna(best_min_rr) else float(best_min_rr),
}
show_selected_params(best_params, title='Final Selected Params')

## 6. Final Report - Read The Result Like A Trader/Risk Manager

This cell reruns the full backtest with `best_params` and renders the final dashboard.

How to read the result:

- KPI: check return, max DD, PF, Sharpe, win rate, trade count, and expectancy.
- Monthly PnL: check whether profit comes from a few unusual months or is distributed across regimes.
- Equity dashboard: look for stagnation, cliff drawdown, and recovery time.
- Trade distribution: inspect tail loss, tail win, skew, and dependence on outliers.
- Trade explorer/chart: verify whether entries and exits make market-structure sense or look noisy.

If the final report does not meet real risk criteria, do not promote it to portfolio work even if `best_params` was selected.


In [ ]:
# Cell 8 - Final full validation report
#
# DOC_FINAL_REPORT_USAGE
#

validation = run_symbol_backtest(
    RUN_CONFIG['symbol'],
    init_eq=RUN_CONFIG['initial_balance'],
    account_mode=RUN_CONFIG['account_mode'],
    date_from=RUN_CONFIG['date_from'],
    date_to=RUN_CONFIG['date_to'],
    max_bars=RUN_CONFIG['max_bars'],
    indicator_overrides=RUN_CONFIG.get('indicator_overrides') or None,
    strategy_overrides=RUN_CONFIG.get('strategy_overrides') or None,
    costs=RUN_CONFIG.get('costs') or None,
    broker_profile=RUN_CONFIG.get('broker_profile'),
    symbol_overrides=best_params,
)

show_kpi_dashboard(validation.metrics, title='Final Validation KPI')
show_monthly_pnl(validation.metrics, title='Final Monthly PnL')
plot_equity_dashboard(validation.equity, validation.trades, title='Final Equity Dashboard')
plot_trade_distribution(validation.trades)
show_trade_explorer(validation.trades, title='Final Trade Explorer')
plot_price_with_trades(validation.signal_data, validation.trades, symbol=RUN_CONFIG['symbol'])


## 7. Optional Walk-Forward - Time-Based Out-Of-Sample Check

Set `RUN_CONFIG['enable_walkforward']=True` to test whether parameters survive across multiple market windows.

Meaning:

- IS window simulates the research/optimization period.
- OOS window simulates the following deployment period.
- Good results should hold across many OOS windows, not just one large winning window.

If OOS consistently underperforms IS, that suggests overfitting or regime dependence.


In [ ]:
# Cell 9 - Optional walk-forward robustness
#
# DOC_WALKFORWARD_OPTIONAL
#

if RUN_CONFIG.get('enable_walkforward'):
    sym_cfg = {**get_symbol_params(RUN_CONFIG['symbol'], broker_profile=RUN_CONFIG.get('broker_profile')), **best_params}
    raw = load_backtest_full(
        sym_cfg['symbol_id'],
        max_bars=RUN_CONFIG['max_bars'],
    )
    ind_params = {
        **get_indicator_params(),
        **(RUN_CONFIG.get('indicator_overrides') or {}),
        'MA_PERIOD': 20,
        'KTP': float(best_params['ktp']),
        'X': float(best_params['x']),
    }
    df_ind = add_combo_indicators(raw.copy(), ind_params)
    wf_df, wf_summary = walk_forward_backtest(
        RUN_CONFIG['symbol'],
        df_ind,
        sym_cfg,
        init_eq=RUN_CONFIG['initial_balance'],
        is_bars=RUN_CONFIG['walkforward_is_bars'],
        oos_bars=RUN_CONFIG['walkforward_oos_bars'],
        step_bars=RUN_CONFIG['walkforward_step_bars'],
        strategy=RUN_CONFIG.get('strategy_overrides') or None,
        costs=RUN_CONFIG.get('costs') or None,
        broker_profile=RUN_CONFIG.get('broker_profile'),
    )
    display(pd.DataFrame([wf_summary]).T.rename(columns={0: 'value'}))
    summarize_walkforward(wf_df)
    plot_walkforward_dashboard(wf_df)
else:
    print('Walk-forward disabled. Set RUN_CONFIG[\'enable_walkforward\'] = True to run it.')

## 8. Optional Monte Carlo - Trade-Sequence Risk Check

Set `RUN_CONFIG['enable_monte_carlo']=True` after full validation has a large enough trade log.

Monte Carlo does not create new edge. It reshuffles/simulates the PnL sequence to estimate:

- Probability of exceeding the drawdown threshold.
- Sharpe confidence interval.
- Impact of win/loss ordering on equity.

If the probability of exceeding DD is high, reduce risk per trade, reduce allocation, add portfolio caps, or discard the candidate.


In [ ]:
# Cell 10 - Optional Monte Carlo trade-order robustness
#
# DOC_MONTE_CARLO_OPTIONAL
#

if RUN_CONFIG.get('enable_monte_carlo'):
    trades_df = pd.DataFrame(validation.trades)
    if trades_df.empty or 'pnl_usd' not in trades_df:
        print('No trade PnL for Monte Carlo.')
    else:
        years = max((validation.equity.index[-1] - validation.equity.index[0]).days / 365.25, 0.1)
        trades_per_year = len(trades_df) / years
        mc = run_monte_carlo(
            trades_df['pnl_usd'].astype(float).tolist(),
            n_iter=RUN_CONFIG['monte_carlo_iter'],
            dd_threshold=RUN_CONFIG['monte_carlo_dd_threshold'],
            initial_balance=RUN_CONFIG['initial_balance'],
            trades_per_year=trades_per_year,
            random_seed=42,
        )
        display(pd.DataFrame({
            'metric': ['prob_exceed_dd', 'sharpe_ci_low', 'sharpe_ci_high', 'trades_per_year'],
            'value': [mc['prob_exceed_dd'], mc['sharpe_ci_low'], mc['sharpe_ci_high'], trades_per_year],
        }))
        plot_monte_carlo(mc)
else:
    print('Monte Carlo disabled. Set RUN_CONFIG[\'enable_monte_carlo\'] = True to run it.')


## 9. Export - Save Research Evidence And Next Steps

Set `RUN_CONFIG['export_report']=True` to save results to the report folder.

Main files:

- `grid.csv`: full fast grid.
- `validated_top.csv`: top candidates after full validation.
- `selected_params.json`: `RUN_CONFIG`, `best_params`, and run metadata.

Next step after export: move `best_params` to the symbol backtest or portfolio notebook, then evaluate correlation, exposure overlap, allocation, max portfolio DD, and forward testing.


In [ ]:
# Cell 11 - Export bundle
#
# DOC_EXPORT_USAGE
#

if RUN_CONFIG.get('export_report'):
    out = export_result_bundle(
        f"{RUN_CONFIG['symbol']}_{RUN_CONFIG['account_mode']}_optimize_research",
        metrics=validation.metrics,
        trades=validation.trades,
        equity=validation.equity,
    )
    grid.to_csv(out / 'grid.csv', index=False, encoding='utf-8-sig')
    filtered_ranked.to_csv(out / 'grid_filtered_ranked.csv', index=False, encoding='utf-8-sig')
    validation_compare.to_csv(out / 'validated_top.csv', index=False, encoding='utf-8-sig')
    metadata = {
        'run_config': RUN_CONFIG,
        'selected_params': best_params,
        'created_at': datetime.utcnow().isoformat() + 'Z',
    }
    with open(out / 'selected_params.json', 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2, default=str)
    print('Exported:', out)
else:
    print("Export disabled. Set RUN_CONFIG['export_report'] = True to save CSV/JSON.")
